# 3.11 — Kernel Ridge Regression

Kernel ridge regression (KRR) is ridge regression written in terms of similarities between training examples instead of explicit feature columns. The payoff is practical: once we can build a kernel matrix `K`, the model solves a regularized linear system for example weights `alpha`, then predicts a new point with a weighted sum of its kernel similarities to the training set.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build kernel ridge regression one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is visible, including the kernel matrix, the regularized solve, and the validation score. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, linear algebra, distances, and numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for the tiny toy data.

### 1. A nonlinear regression problem and the ridge question

Kernel ridge starts with the ordinary ERM question: choose predictions that make the average loss small, but do not let flexibility run wild. We use a one-dimensional dataset with a curved target so a straight line is visibly too small a hypothesis family. The regularization question is already present: should the model chase every bump, or should it prefer a smoother rule that is likely to survive future data?

In [ ]:
X_w = np.linspace(-3, 3, 9)[:, None]  # nine training inputs as a column vector.
y_w = np.sin(X_w[:, 0]) + 0.18 * X_w[:, 0]  # a smooth nonlinear target with a slight trend.
print("X shape:", X_w.shape, "y shape:", y_w.shape)  # inspect training shapes.
print("first three pairs:", np.round(np.c_[X_w[:3, 0], y_w[:3]], 3))  # inspect concrete data.

▶ What you'll see: a small table of input-output pairs; the outputs curve rather than lying on one straight line.

In [ ]:
plt.figure(figsize=(4.6, 3))  # draw the training set.
plt.scatter(X_w[:, 0], y_w, color="black", label="training data")  # observed examples.
plt.axhline(0, color="gray", linewidth=0.6)  # reference line.
plt.title("1: nonlinear training signal")  # title the plot.
plt.xlabel("x"); plt.ylabel("y")  # label axes.
plt.legend(); plt.show()  # display the figure.

▶ What you'll see: the points follow a smooth S-shaped curve, so a similarity-based nonlinear smoother is plausible.

*Why it's done this way:* KRR keeps the empirical-risk frame from earlier lessons: fit the observed targets, then pay a complexity cost. The nonlinear shape tells us that explicit straight-line features are not the point; the method will instead compare examples through a kernel and solve ridge in that similarity space.

### 2. The kernel matrix stores pairwise similarities

A kernel is a function `k(x, z)` that behaves like a dot product in some feature space, even when we never write those features down. For the Gaussian/RBF kernel,

$$k(x,z)=\exp(-\gamma\lVert x-z\rVert^2),$$

nearby points receive similarity close to 1, and distant points receive similarity close to 0. Evaluating that kernel on every pair of training points gives the Gram matrix `K`; row `i` says how example `i` sees all other examples.

In [ ]:
def rbf_kernel_w(A_w, B_w, gamma_w):  # compute all pairwise RBF similarities.
    A_w = np.asarray(A_w, dtype=float)  # ensure a numeric 2-D array.
    B_w = np.asarray(B_w, dtype=float)  # ensure a numeric 2-D array.
    sqdist_w = np.sum((A_w[:, None, :] - B_w[None, :, :]) ** 2, axis=2)  # ||a-b||^2 for every pair.
    return np.exp(-gamma_w * sqdist_w)  # convert distances to similarities.

gamma_w = 0.7  # bandwidth knob: larger means more local similarities.
K_w = rbf_kernel_w(X_w, X_w, gamma_w)  # training-by-training kernel matrix.
print("K shape:", K_w.shape)  # one row/column per training example.
print("top-left block:\n", np.round(K_w[:3, :3], 3))  # inspect local similarities.
assert np.allclose(np.diag(K_w), 1.0)  # every point is maximally similar to itself.

▶ What you'll see: diagonal entries are 1, while off-diagonal entries shrink with distance.

In [ ]:
plt.figure(figsize=(4.2, 3.2))  # heatmap of similarities.
plt.imshow(K_w, cmap="viridis")  # color encodes kernel similarity.
plt.colorbar(label="k(x_i, x_j)")  # similarity scale.
plt.title("2: RBF kernel matrix K")  # title the heatmap.
plt.xlabel("training index j"); plt.ylabel("training index i")  # matrix axes.
plt.show()  # display the figure.

▶ What you'll see: a bright diagonal band — neighboring x-values are similar, far-away x-values are not.

*Why it's done this way:* the kernel matrix is the only feature representation KRR needs. If a model prediction is a weighted sum of training-example similarities, then `K` is the design matrix for those weights on the training set.

### 3. Ridge regularization makes the solve stable

The core KRR training equation from the lesson is

$$\alpha=(K+\lambda I)^{-1}y.$$

The vector `alpha` contains one weight per training example. Adding `lambda I` is ridge regularization: it keeps the linear system invertible and discourages extreme weights, especially when two training examples are nearly redundant under the kernel.

In [ ]:
lam_w = 0.12  # ridge penalty; larger values make alpha smaller and the function smoother.
A_w = K_w + lam_w * np.eye(len(X_w))  # regularized kernel system.
alpha_w = np.linalg.solve(A_w, y_w)  # solve instead of explicitly inverting.
print("alpha:", np.round(alpha_w, 3))  # inspect example weights.
print("condition K:", round(np.linalg.cond(K_w), 1), "condition K+lambda I:", round(np.linalg.cond(A_w), 1))  # stability check.
assert np.linalg.cond(A_w) < np.linalg.cond(K_w)  # ridge improves conditioning here.

▶ What you'll see: the regularized system has a much smaller condition number than raw `K`.

In [ ]:
fit_w = K_w @ alpha_w  # training predictions from similarities to the training set.
losses_w = (fit_w - y_w) ** 2  # per-example squared errors.
print("per-example losses:", np.round(losses_w[:4], 4))  # inspect empirical losses.
print("empirical risk:", round(float(losses_w.mean()), 4))  # average training loss.

▶ What you'll see: training predictions are close to the targets but not forced to interpolate exactly.

*Why it's done this way:* solving `(K + λI) alpha = y` is ridge regression in dual form. The `λI` term is not a coding trick; it is the mathematical cost that trades a slightly worse raw fit for a more stable, smaller-weight solution.

### 4. Predictions are weighted sums of kernel similarities

After training, a new input `x` does not need explicit polynomial or radial features. We compute its similarity to each training input, stack those values into `k(x, X)`, and take the dot product with `alpha`:

$$\hat y(x)=k(x,X)^\top\alpha.$$

Every training example casts a vote, and the kernel decides how much that vote matters at the new location.

In [ ]:
x_grid_w = np.linspace(-3.4, 3.4, 160)[:, None]  # dense x values for plotting predictions.
K_grid_w = rbf_kernel_w(x_grid_w, X_w, gamma_w)  # new-by-training similarities.
y_grid_w = K_grid_w @ alpha_w  # KRR predictions on the grid.
print("prediction grid shape:", y_grid_w.shape)  # one prediction per grid point.
print("middle prediction:", round(float(y_grid_w[len(y_grid_w)//2]), 3))  # inspect one value.

▶ What you'll see: the model produces a dense vector of predictions from a dense matrix of similarities.

In [ ]:
plt.figure(figsize=(5, 3))  # visualize fitted function.
plt.plot(x_grid_w[:, 0], y_grid_w, color="seagreen", label="KRR fit")  # smooth prediction curve.
plt.scatter(X_w[:, 0], y_w, color="black", s=24, label="training data")  # original examples.
plt.title("4: prediction as k(x, X)^T alpha")  # title the plot.
plt.xlabel("x"); plt.ylabel("prediction")  # label axes.
plt.legend(); plt.show()  # display the figure.

▶ What you'll see: the curve bends with the data while remaining smooth between examples.

*Why it's done this way:* the prediction formula is the representer idea in action: the learned function lives in the span of kernel bumps centered at the training examples, so a new prediction is just a similarity-weighted sum of learned example weights.

### 5. The bandwidth knob controls locality

The RBF parameter `gamma` controls how quickly similarity decays with distance. Small `gamma` makes broad kernels: many points influence each other, producing a smoother global curve. Large `gamma` makes narrow kernels: only very nearby points influence each prediction, which can chase local variation.

In [ ]:
gammas_w = [0.15, 0.7, 4.0]  # broad, medium, and local kernels.
curves_w = []  # store one fitted curve per gamma.
for g_w in gammas_w:  # fit KRR at each bandwidth.
    K_g_w = rbf_kernel_w(X_w, X_w, g_w)  # training kernel for this gamma.
    alpha_g_w = np.linalg.solve(K_g_w + lam_w * np.eye(len(X_w)), y_w)  # regularized dual weights.
    curves_w.append(rbf_kernel_w(x_grid_w, X_w, g_w) @ alpha_g_w)  # predictions on the grid.
print("curve values at x≈0:", [round(float(c[len(c)//2]), 3) for c in curves_w])  # compare a shared location.

▶ What you'll see: all three curves score the center similarly, but they will differ most between and beyond points.

In [ ]:
plt.figure(figsize=(5.2, 3.2))  # compare bandwidths.
for g_w, curve_w in zip(gammas_w, curves_w):  # plot each curve.
    plt.plot(x_grid_w[:, 0], curve_w, label=f"gamma={g_w}")  # bandwidth-labeled line.
plt.scatter(X_w[:, 0], y_w, color="black", s=20)  # training data.
plt.title("5: bandwidth controls locality")  # title the plot.
plt.xlabel("x"); plt.ylabel("prediction")  # label axes.
plt.legend(); plt.show()  # display the figure.

▶ What you'll see: the small-gamma curve is broad and smooth; the large-gamma curve bends more locally around points.

*Why it's done this way:* `gamma` defines what counts as "near." Because KRR predictions are similarity-weighted sums, changing the similarity decay changes the effective hypothesis family just as surely as changing model architecture would.

### 6. Validation score means raw fit plus model cost

The lesson text emphasizes a contract: raw empirical loss is only one term, and selection should include a penalty, cost, or validation contrast. We therefore compute training loss, validation loss, a regularization-cost proxy, and a final decision score. The smaller score is not automatically the most flexible curve; it is the curve whose gain survives its cost.

In [ ]:
X_val_w = np.array([[-2.6], [-1.1], [0.4], [1.9], [2.8]])  # held-out locations.
y_val_w = np.sin(X_val_w[:, 0]) + 0.18 * X_val_w[:, 0]  # held-out targets from the same signal.
settings_w = [(0.15, 0.12), (0.7, 0.12), (4.0, 0.12)]  # candidate bandwidths with fixed ridge.
scores_w = []  # store validation-plus-cost scores.
for g_w, l_w in settings_w:  # evaluate each candidate.
    K_train_w = rbf_kernel_w(X_w, X_w, g_w)  # train kernel.
    alpha_s_w = np.linalg.solve(K_train_w + l_w * np.eye(len(X_w)), y_w)  # fit weights.
    val_pred_w = rbf_kernel_w(X_val_w, X_w, g_w) @ alpha_s_w  # validation predictions.
    val_mse_w = float(np.mean((val_pred_w - y_val_w) ** 2))  # raw held-out fit.
    cost_w = 0.01 * float(alpha_s_w @ K_train_w @ alpha_s_w)  # small RKHS-norm-style complexity cost.
    scores_w.append(val_mse_w + cost_w)  # final decision score.
print("decision scores:", np.round(scores_w, 4))  # inspect full score, not raw fit alone.

▶ What you'll see: each candidate gets a single comparable number after adding a cost to validation loss.

In [ ]:
best_w = int(np.argmin(scores_w))  # choose the lowest full decision score.
plt.figure(figsize=(5, 3))  # bar chart of decision scores.
plt.bar([f"γ={g}" for g, _ in settings_w], scores_w, color=["gray" if i != best_w else "seagreen" for i in range(len(scores_w))])  # highlight winner.
plt.title("6: select by validation loss plus cost")  # title the plot.
plt.ylabel("decision score")  # lower is better.
plt.show()  # display the comparison.
print("selected gamma:", settings_w[best_w][0])  # report chosen bandwidth.

▶ What you'll see: the selected setting is the one with the lowest full score, not merely the most wiggly fit.

*Why it's done this way:* the mathematics block's average-plus-cost logic is the selection rule. KRR has two flexibility knobs (`gamma` and `lambda`), so the validation score protects us from confusing an attractive training curve with a durable model.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each KRR mechanic by hand.** These new tiny NumPy-only toys isolate
> nonlinear data, kernel similarities, ridge stabilization, kernel predictions, bandwidth locality,
> and validation-plus-cost selection. Run them top to bottom: every block prints intermediates,
> draws one picture, and includes an `assert`.

### ✍️ Toy 1 · A straight line misses a curved pattern

Before kernels, inspect a small curved dataset and the best straight-line baseline. The residual error motivates a nonlinear smoother.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_x = np.array([-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0])
print("x values:", t1_x.tolist())  # -> [-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
t1_y = np.array([2.0, 0.0, -1.0, 0.0, -1.0, 0.0, 2.0])
print("curved targets:", t1_y.tolist())  # -> [2.0, 0.0, -1.0, 0.0, -1.0, 0.0, 2.0]
t1_design = np.column_stack([np.ones_like(t1_x), t1_x])
print("linear design shape:", t1_design.shape)  # -> (7, 2)
t1_normal = t1_design.T @ t1_design
print("normal matrix:", t1_normal.tolist())  # -> [[7.0, 0.0], [0.0, 28.0]]
t1_rhs = t1_design.T @ t1_y
print("right-hand side:", t1_rhs.tolist())  # -> [2.0, 0.0]
t1_beta = np.linalg.solve(t1_normal, t1_rhs)  # -> [0.28571429 0.        ]
print("best line beta:", np.round(t1_beta, 3).tolist())  # -> [0.286, 0.0]
t1_pred = t1_design @ t1_beta  # -> [0.28571429 0.28571429 0.28571429 0.28571429 0.28571429 0.28571429 0.28571429]
print("line predictions:", np.round(t1_pred, 3).tolist())  # -> [0.286, 0.286, 0.286, 0.286, 0.286, 0.286, 0.286]
t1_mse = float(np.mean((t1_y - t1_pred) ** 2))  # -> 1.346938775510204
print("line MSE:", round(t1_mse, 3))  # -> 1.347
assert round(t1_mse, 3) == 1.347

plt.figure(figsize=(5.0, 2.8))
plt.scatter(t1_x, t1_y, color="black", label="curved data")
plt.plot(t1_x, t1_pred, color="crimson", label="best line")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Toy 1 · linear baseline underfits")
plt.legend()
plt.show()

▶ What you'll see: the line is flat while the data curve down and back up.

### ✍️ Toy 2 · The RBF kernel matrix stores pairwise similarity

Pairwise squared distances become similarities. The diagonal is exactly one because every point is identical to itself.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_X = np.array([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0])[:, None]
print("training inputs:", t2_X[:, 0].tolist())  # -> [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
t2_gamma = 0.5
print("gamma:", t2_gamma)  # -> 0.5
t2_sqdist = np.sum((t2_X[:, None, :] - t2_X[None, :, :]) ** 2, axis=2)
print("top-left squared distances:", t2_sqdist[:3, :3].astype(int).tolist())  # -> [[0, 1, 4], [1, 0, 1], [4, 1, 0]]
t2_K = np.exp(-t2_gamma * t2_sqdist)
print("first kernel row:", np.round(t2_K[0], 3).tolist())  # -> [1.0, 0.607, 0.135, 0.011, 0.0, 0.0]
t2_diag = np.diag(t2_K)
print("kernel diagonal:", np.round(t2_diag, 3).tolist())  # -> [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
assert np.allclose(t2_diag, 1.0)

plt.figure(figsize=(4.2, 3.0))
plt.imshow(t2_K, cmap="viridis")
plt.colorbar(label="similarity")
plt.title("Toy 2 · RBF kernel matrix")
plt.xlabel("j")
plt.ylabel("i")
plt.show()

▶ What you'll see: a bright diagonal band where nearby training points are most similar.

### ✍️ Toy 3 · Adding lambda stabilizes the kernel solve

Solve `(K + λI) alpha = y`. The ridge term improves conditioning and keeps the example weights finite.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_X = np.array([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0])[:, None]
print("training inputs:", t3_X[:, 0].tolist())  # -> [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
t3_y = np.array([0.0, 1.0, 0.0, -1.0, 0.0, 1.0])
print("targets:", t3_y.tolist())  # -> [0.0, 1.0, 0.0, -1.0, 0.0, 1.0]
t3_gamma = 0.7
print("gamma:", t3_gamma)  # -> 0.7
t3_sqdist = np.sum((t3_X[:, None, :] - t3_X[None, :, :]) ** 2, axis=2)
t3_K = np.exp(-t3_gamma * t3_sqdist)
print("kernel first row:", np.round(t3_K[0], 3).tolist())  # -> [1.0, 0.497, 0.061, 0.002, 0.0, 0.0]
t3_lam = 0.25
print("lambda:", t3_lam)  # -> 0.25
t3_A = t3_K + t3_lam * np.eye(len(t3_X))
t3_cond_K = float(np.linalg.cond(t3_K))  # -> 10.664269516503568
print("condition K:", round(t3_cond_K, 1))  # -> 10.7
t3_cond_A = float(np.linalg.cond(t3_A))  # -> 5.077619197774036
print("condition K+lambda I:", round(t3_cond_A, 1))  # -> 5.1
t3_alpha = np.linalg.solve(t3_A, t3_y)  # -> [-0.40067918  1.01576742 -0.03647509 -0.88326278  0.01919219  0.83466692]
print("alpha weights:", np.round(t3_alpha, 3).tolist())  # -> [-0.401, 1.016, -0.036, -0.883, 0.019, 0.835]
assert t3_cond_A < t3_cond_K

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t3_alpha.size), t3_alpha, color="steelblue")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("training example")
plt.ylabel("alpha")
plt.title("Toy 3 · regularized example weights")
plt.show()

▶ What you'll see: the regularized system has a smaller condition number and finite alpha weights.

### ✍️ Toy 4 · Predictions are similarity-weighted sums

For each new point, compute its similarities to training points and dot them with the learned alpha weights.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_X = np.array([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0])[:, None]
print("training inputs:", t4_X[:, 0].tolist())  # -> [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
t4_y = np.array([0.0, 1.0, 0.0, -1.0, 0.0, 1.0])
print("targets:", t4_y.tolist())  # -> [0.0, 1.0, 0.0, -1.0, 0.0, 1.0]
t4_new = np.array([-1.5, 0.5, 2.5])[:, None]
print("new inputs:", t4_new[:, 0].tolist())  # -> [-1.5, 0.5, 2.5]
t4_gamma = 0.7
print("gamma:", t4_gamma)  # -> 0.7
t4_lam = 0.25
print("lambda:", t4_lam)  # -> 0.25
t4_K = np.exp(-t4_gamma * np.sum((t4_X[:, None, :] - t4_X[None, :, :]) ** 2, axis=2))
t4_alpha = np.linalg.solve(t4_K + t4_lam * np.eye(len(t4_X)), t4_y)
print("alpha weights:", np.round(t4_alpha, 3).tolist())  # -> [-0.401, 1.016, -0.036, -0.883, 0.019, 0.835]
t4_Knew = np.exp(-t4_gamma * np.sum((t4_new[:, None, :] - t4_X[None, :, :]) ** 2, axis=2))
print("first new-point similarities:", np.round(t4_Knew[0], 3).tolist())  # -> [0.839, 0.839, 0.207, 0.013, 0.0, 0.0]
t4_pred = t4_Knew @ t4_alpha  # -> [ 0.49808288 -0.55069645  0.53405291]
print("new predictions:", np.round(t4_pred, 3).tolist())  # -> [0.498, -0.551, 0.534]
assert np.allclose(np.round(t4_pred, 3), [0.498, -0.551, 0.534])

plt.figure(figsize=(5.0, 2.8))
plt.scatter(t4_X[:, 0], t4_y, color="black", label="train")
plt.scatter(t4_new[:, 0], t4_pred, color="crimson", label="KRR predictions")
plt.xlabel("x")
plt.ylabel("prediction")
plt.title("Toy 4 · k(x, X)ᵀ alpha")
plt.legend()
plt.show()

▶ What you'll see: new red points are weighted blends of nearby training-example votes.

### ✍️ Toy 5 · Gamma controls locality

Small `gamma` keeps similarities broad; large `gamma` makes only nearest neighbors matter, changing the fitted curve.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_X = np.array([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0])[:, None]
print("training inputs:", t5_X[:, 0].tolist())  # -> [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
t5_y = np.array([0.0, 1.0, 0.0, -1.0, 0.0, 1.0])
print("targets:", t5_y.tolist())  # -> [0.0, 1.0, 0.0, -1.0, 0.0, 1.0]
t5_grid = np.linspace(-2.0, 3.0, 6)[:, None]
print("plot grid:", t5_grid[:, 0].tolist())  # -> [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
t5_gammas = np.array([0.15, 1.0, 4.0])
print("gamma grid:", t5_gammas.tolist())  # -> [0.15, 1.0, 4.0]
t5_lam = 0.2
print("lambda:", t5_lam)  # -> 0.2
t5_curves = []
t5_mid_predictions = []
t5_neighbor_sims = []
for t5_gamma in t5_gammas:
    t5_sqdist = np.sum((t5_X[:, None, :] - t5_X[None, :, :]) ** 2, axis=2)
    t5_K = np.exp(-t5_gamma * t5_sqdist)
    t5_alpha = np.linalg.solve(t5_K + t5_lam * np.eye(len(t5_X)), t5_y)
    t5_Kgrid = np.exp(-t5_gamma * np.sum((t5_grid[:, None, :] - t5_X[None, :, :]) ** 2, axis=2))
    t5_curve = t5_Kgrid @ t5_alpha
    t5_curves.append(t5_curve)
    t5_mid_predictions.append(float(t5_curve[2]))
    t5_neighbor_sims.append(float(np.exp(-t5_gamma * 1.0)))
t5_curves = np.array(t5_curves)
print("distance-1 similarities:", np.round(t5_neighbor_sims, 3).tolist())  # -> [0.861, 0.368, 0.018]
print("predictions at x=0:", np.round(t5_mid_predictions, 3).tolist())  # -> [-0.055, 0.004, 0.0]
print("curves:", np.round(t5_curves, 3).tolist())  # -> [[0.348, 0.38, -0.055, -0.385, -0.05, 0.64], [0.058, 0.812, 0.004, -0.829, -0.001, 0.831], [0.003, 0.833, 0.0, -0.833, -0.0, 0.833]]
assert t5_neighbor_sims[2] < t5_neighbor_sims[1] < t5_neighbor_sims[0]

plt.figure(figsize=(5.2, 3.0))
for t5_gamma, t5_curve in zip(t5_gammas, t5_curves):
    plt.plot(t5_grid[:, 0], t5_curve, marker="o", label=f"γ={t5_gamma}")
plt.scatter(t5_X[:, 0], t5_y, color="black", s=20, label="train")
plt.xlabel("x")
plt.ylabel("prediction")
plt.title("Toy 5 · bandwidth changes locality")
plt.legend()
plt.show()

▶ What you'll see: the large-gamma curve tracks individual points more locally than the broad small-gamma curve.

### ✍️ Toy 6 · Validation score adds a complexity cost

Evaluate candidate `(gamma, lambda)` settings on held-out points, add a small norm-style cost, and choose the lowest full score.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_X = np.array([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0])[:, None]
print("training inputs:", t6_X[:, 0].tolist())  # -> [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
t6_y = np.array([0.0, 1.0, 0.0, -1.0, 0.0, 1.0])
print("training targets:", t6_y.tolist())  # -> [0.0, 1.0, 0.0, -1.0, 0.0, 1.0]
t6_Xval = np.array([-1.5, -0.5, 0.5, 1.5, 2.5])[:, None]
print("validation inputs:", t6_Xval[:, 0].tolist())  # -> [-1.5, -0.5, 0.5, 1.5, 2.5]
t6_yval = np.array([0.5, 0.5, -0.5, -0.5, 0.5])
print("validation targets:", t6_yval.tolist())  # -> [0.5, 0.5, -0.5, -0.5, 0.5]
t6_settings = np.array([[0.2, 0.1], [0.7, 0.25], [3.0, 0.25]])
print("settings [gamma, lambda]:", t6_settings.tolist())  # -> [[0.2, 0.1], [0.7, 0.25], [3.0, 0.25]]
t6_mses = []
t6_costs = []
t6_scores = []
for t6_gamma, t6_lam in t6_settings:
    t6_K = np.exp(-t6_gamma * np.sum((t6_X[:, None, :] - t6_X[None, :, :]) ** 2, axis=2))
    t6_alpha = np.linalg.solve(t6_K + t6_lam * np.eye(len(t6_X)), t6_y)
    t6_Kval = np.exp(-t6_gamma * np.sum((t6_Xval[:, None, :] - t6_X[None, :, :]) ** 2, axis=2))
    t6_pred = t6_Kval @ t6_alpha
    t6_mse = float(np.mean((t6_yval - t6_pred) ** 2))
    t6_cost = 0.02 * float(t6_alpha @ t6_K @ t6_alpha)
    t6_mses.append(t6_mse)
    t6_costs.append(t6_cost)
    t6_scores.append(t6_mse + t6_cost)
t6_mses = np.array(t6_mses)
print("validation MSEs:", np.round(t6_mses, 4).tolist())  # -> [0.0048, 0.0019, 0.0158]
t6_costs = np.array(t6_costs)
print("complexity costs:", np.round(t6_costs, 4).tolist())  # -> [0.104, 0.0413, 0.0384]
t6_scores = np.array(t6_scores)
print("full scores:", np.round(t6_scores, 4).tolist())  # -> [0.1088, 0.0432, 0.0542]
t6_best = int(np.argmin(t6_scores))  # -> 1
print("selected setting:", t6_settings[t6_best].tolist())  # -> [0.7, 0.25]
assert t6_best == 1

plt.figure(figsize=(5.0, 2.8))
plt.bar(["γ=.2", "γ=.7", "γ=3"], t6_scores, color=["gray", "seagreen", "gray"])
plt.ylabel("validation + cost")
plt.title("Toy 6 · choose the full score")
plt.show()

▶ What you'll see: the middle setting has the lowest validation-plus-cost score.

## 🛠️ Setup

In [ ]:
import numpy as np  # Import NumPy for arrays, kernels, linear solves, and assertions.
import matplotlib.pyplot as plt  # Import Matplotlib for inspecting curves, matrices, and score comparisons.
np.random.seed(0)  # Fix the global random seed so all examples are reproducible.

def rbf_kernel(A, B, gamma):  # Compute an RBF kernel matrix between two 2-D arrays.
    A = np.asarray(A, dtype=float)  # Convert the left inputs to numeric arrays.
    B = np.asarray(B, dtype=float)  # Convert the right inputs to numeric arrays.
    sqdist = np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=2)  # Pairwise squared distances.
    return np.exp(-gamma * sqdist)  # Turn distances into similarities.

def krr_fit(X, y, gamma, lam):  # Fit kernel ridge regression in dual form.
    K = rbf_kernel(X, X, gamma)  # Build the training Gram matrix.
    alpha = np.linalg.solve(K + lam * np.eye(len(X)), y)  # Solve (K + λI)α = y.
    return alpha, K  # Return both weights and the training kernel for inspection.

def krr_predict(X_new, X_train, alpha, gamma):  # Predict with trained KRR weights.
    return rbf_kernel(X_new, X_train, gamma) @ alpha  # Compute k(x,X)^T alpha for each new point.

## 🟢 Basics (warm-up)

### Basic 1 — Create a curved toy dataset

**Goal.** Build a tiny nonlinear regression table, because KRR is easiest to understand when a straight line is visibly inadequate. We build it in 2 steps.

In [ ]:
X_b1 = np.linspace(-3, 3, 9)[:, None]  # Create nine one-dimensional inputs as a column vector.
y_b1 = np.sin(X_b1[:, 0]) + 0.18 * X_b1[:, 0]  # Create smooth nonlinear targets with a small trend.
print("shapes:", X_b1.shape, y_b1.shape)  # Inspect the training array shapes.
print("first rows:\n", np.round(np.c_[X_b1[:3, 0], y_b1[:3]], 3))  # Inspect concrete examples.

▶ What you'll see: a small input-output table with negative, zero-ish, and positive regions.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a compact data plot.
plt.scatter(X_b1[:, 0], y_b1, color="black")  # Show the observed examples.
plt.title("Basic 1: curved regression data")  # Title the figure.
plt.xlabel("x"); plt.ylabel("y")  # Label axes.
plt.show()  # Display the plot.

▶ What you'll see: the target bends, which motivates using similarities instead of only a line.

👀 Takeaway: KRR is still supervised regression; the kernel changes the feature geometry, not the basic data table.

### Basic 2 — Compute squared distances

**Goal.** Build pairwise squared distances, because the RBF kernel converts distances into similarities. We build it in 2 steps.

In [ ]:
X_b2 = np.array([[-1.0], [0.0], [2.0]])  # Define three simple one-dimensional points.
diff_b2 = X_b2[:, None, :] - X_b2[None, :, :]  # Broadcast every point against every other point.
print("pairwise differences:\n", diff_b2[:, :, 0])  # Inspect signed differences before squaring.

▶ What you'll see: a 3×3 table where entry i,j is x_i - x_j.

In [ ]:
sqdist_b2 = np.sum(diff_b2 ** 2, axis=2)  # Square and sum coordinate differences.
print("squared distances:\n", sqdist_b2)  # Inspect nonnegative distances.
assert sqdist_b2[0, 2] == 9.0  # (-1 - 2)^2 = 9.
plt.figure(figsize=(4, 3))  # Create a heatmap.
plt.imshow(sqdist_b2, cmap="magma")  # Plot distances as color.
plt.colorbar(label="squared distance")  # Add scale.
plt.title("Basic 2: pairwise squared distances")  # Title the plot.
plt.show()  # Display the heatmap.

▶ What you'll see: the diagonal is zero and farther points have larger squared distances.

👀 Takeaway: RBF kernels start from distance, so broadcasting distances correctly is the first numerical building block.

### Basic 3 — Turn distances into RBF similarities

**Goal.** Apply `exp(-gamma distance²)`, because KRR needs similarity weights that decay with distance. We build it in 2 steps.

In [ ]:
X_b3 = np.array([[-1.0], [0.0], [2.0]])  # Recreate the three-point input set.
gamma_b3 = 0.5  # Choose a moderate RBF bandwidth parameter.
K_b3 = rbf_kernel(X_b3, X_b3, gamma_b3)  # Compute pairwise RBF similarities.
print("RBF kernel:\n", np.round(K_b3, 3))  # Inspect similarity values.
assert np.allclose(np.diag(K_b3), 1.0)  # Each point is exactly similar to itself.

▶ What you'll see: the diagonal is 1 and the farthest pair has the smallest similarity.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a kernel heatmap.
plt.imshow(K_b3, cmap="viridis")  # Visualize similarities.
plt.colorbar(label="similarity")  # Add color scale.
plt.title("Basic 3: RBF kernel matrix")  # Title the plot.
plt.show()  # Display the kernel matrix.

▶ What you'll see: nearby examples form brighter blocks than distant examples.

👀 Takeaway: a kernel matrix is a similarity table that can replace explicit engineered features.

### Basic 4 — Add ridge to the kernel matrix

**Goal.** Compare `K` with `K + λI`, because ridge regularization stabilizes the linear solve. We build it in 2 steps.

In [ ]:
X_b4 = np.linspace(-2, 2, 5)[:, None]  # Create five evenly spaced inputs.
K_b4 = rbf_kernel(X_b4, X_b4, 0.8)  # Compute their RBF kernel matrix.
lam_b4 = 0.2  # Choose a ridge penalty.
A_b4 = K_b4 + lam_b4 * np.eye(len(X_b4))  # Add lambda to the diagonal.
print("diag before:", np.round(np.diag(K_b4), 2))  # Inspect original diagonal.
print("diag after:", np.round(np.diag(A_b4), 2))  # Inspect regularized diagonal.

▶ What you'll see: each diagonal entry increases from 1.0 to 1.2.

In [ ]:
cond_K_b4 = np.linalg.cond(K_b4)  # Measure raw kernel conditioning.
cond_A_b4 = np.linalg.cond(A_b4)  # Measure regularized conditioning.
print("condition numbers:", round(cond_K_b4, 2), "->", round(cond_A_b4, 2))  # Inspect stability improvement.
assert cond_A_b4 < cond_K_b4  # Ridge improves this solve.
plt.figure(figsize=(4, 3))  # Create a comparison plot.
plt.bar(["K", "K+λI"], [cond_K_b4, cond_A_b4], color=["crimson", "seagreen"])  # Compare conditioning.
plt.title("Basic 4: ridge improves conditioning")  # Title the plot.
plt.ylabel("condition number")  # Label y-axis.
plt.show()  # Display the bars.

▶ What you'll see: the regularized system is numerically easier to solve.

👀 Takeaway: λ is the stabilizing cost that prevents the kernel solve from becoming brittle.

### Basic 5 — Solve for alpha weights

**Goal.** Compute `alpha = solve(K + λI, y)`, because KRR learns one dual weight per training example. We build it in 2 steps.

In [ ]:
X_b5 = np.linspace(-2, 2, 5)[:, None]  # Create training inputs.
y_b5 = np.sin(X_b5[:, 0])  # Create nonlinear targets.
alpha_b5, K_b5 = krr_fit(X_b5, y_b5, gamma=0.8, lam=0.2)  # Fit KRR dual weights.
print("alpha:", np.round(alpha_b5, 3))  # Inspect example weights.
print("alpha shape:", alpha_b5.shape)  # Confirm one weight per training point.

▶ What you'll see: five alpha values, matching the five training examples.

In [ ]:
fit_b5 = K_b5 @ alpha_b5  # Compute training predictions.
print("train fit:", np.round(fit_b5, 3))  # Inspect predictions at training inputs.
print("targets:", np.round(y_b5, 3))  # Compare with true targets.
plt.figure(figsize=(4, 3))  # Plot fit versus targets.
plt.scatter(X_b5[:, 0], y_b5, color="black", label="target")  # True values.
plt.scatter(X_b5[:, 0], fit_b5, color="seagreen", label="KRR fit")  # Fitted values.
plt.title("Basic 5: alpha produces fitted values")  # Title plot.
plt.legend(); plt.show()  # Display with legend.

▶ What you'll see: fitted values are close but gently shrunk toward smoother behavior.

👀 Takeaway: alpha weights are not feature coefficients; they weight training examples through the kernel.

### Basic 6 — Predict a new point

**Goal.** Score one new input with `k(x, X)^T alpha`, because prediction uses similarities to the training set. We build it in 2 steps.

In [ ]:
X_b6 = np.linspace(-2, 2, 5)[:, None]  # Define training inputs.
y_b6 = np.sin(X_b6[:, 0])  # Define training targets.
alpha_b6, K_b6 = krr_fit(X_b6, y_b6, gamma=0.8, lam=0.2)  # Fit dual weights.
x_new_b6 = np.array([[0.5]])  # Choose one new input.
k_new_b6 = rbf_kernel(x_new_b6, X_b6, 0.8)  # Similarities from new input to training examples.
print("new similarities:", np.round(k_new_b6[0], 3))  # Inspect voting weights.

▶ What you'll see: training points near 0.5 receive larger similarity weights.

In [ ]:
pred_b6 = float(k_new_b6 @ alpha_b6)  # Compute the KRR prediction for the new point.
print("prediction at 0.5:", round(pred_b6, 3))  # Inspect numeric prediction.
assert abs(pred_b6 - float(krr_predict(x_new_b6, X_b6, alpha_b6, 0.8))) < 1e-12  # Verify helper formula.
plt.figure(figsize=(4, 3))  # Plot similarity votes.
plt.bar([str(float(x)) for x in X_b6[:, 0]], k_new_b6[0], color="teal")  # Similarity to each training x.
plt.title("Basic 6: similarities for x=0.5")  # Title plot.
plt.ylabel("k(x, X_i)")  # Label axis.
plt.show()  # Display bars.

▶ What you'll see: the nearest training points cast the strongest votes.

👀 Takeaway: KRR predictions are weighted sums of training-example influence.

### Basic 7 — Compute empirical risk

**Goal.** Average squared errors on training examples, because ERM judges a model by mean loss before adding cost or validation. We build it in 2 steps.

In [ ]:
X_b7 = np.linspace(-2, 2, 5)[:, None]  # Create inputs.
y_b7 = np.sin(X_b7[:, 0])  # Create targets.
alpha_b7, K_b7 = krr_fit(X_b7, y_b7, gamma=0.8, lam=0.2)  # Fit KRR.
fit_b7 = K_b7 @ alpha_b7  # Training predictions.
losses_b7 = (fit_b7 - y_b7) ** 2  # Per-example squared losses.
print("losses:", np.round(losses_b7, 5))  # Inspect individual losses.

▶ What you'll see: each training example contributes one nonnegative loss.

In [ ]:
risk_b7 = float(np.mean(losses_b7))  # Average the per-example losses.
print("empirical risk:", round(risk_b7, 5))  # Inspect ERM quantity.
assert risk_b7 >= 0.0  # Squared-error risk cannot be negative.
plt.figure(figsize=(4, 3))  # Plot losses.
plt.bar(range(len(losses_b7)), losses_b7, color="orange")  # One bar per example.
plt.title("Basic 7: per-example squared losses")  # Title plot.
plt.ylabel("loss")  # Label y-axis.
plt.show()  # Display bars.

▶ What you'll see: the empirical risk is the average height of the loss bars.

👀 Takeaway: KRR still optimizes average prediction loss; the kernel only changes the function class.

### Basic 8 — Add a method cost to the score

**Goal.** Reproduce the lesson's raw-loss-plus-cost logic, because selection should not use raw fit alone. We build it in 2 steps.

In [ ]:
losses_b8 = np.array([0.202, 0.070, 0.420])  # Use the verified toy losses from the lesson text.
risk_b8 = float(np.mean(losses_b8))  # Average the empirical losses.
cost_b8 = 0.100  # Use the lesson's stabilizing or operational cost.
print("risk:", round(risk_b8, 3), "cost:", round(cost_b8, 3))  # Inspect both pieces.
assert round(risk_b8, 3) == 0.231  # Verify the lesson average.

▶ What you'll see: the raw empirical risk is 0.231 before cost is added.

In [ ]:
score_b8 = risk_b8 + cost_b8  # Add method cost to raw risk.
print("decision score:", round(score_b8, 3))  # Inspect the selection score.
assert round(score_b8, 3) == 0.331  # Verify the lesson score.
plt.figure(figsize=(4, 3))  # Plot score pieces.
plt.bar(["risk", "cost", "total"], [risk_b8, cost_b8, score_b8], color=["steelblue", "gray", "seagreen"])  # Show additive score.
plt.title("Basic 8: risk plus cost")  # Title plot.
plt.show()  # Display bars.

▶ What you'll see: the total score is larger than raw training loss because flexibility has a price.

👀 Takeaway: the unit of comparison is the full decision score, not the prettiest raw loss.

### Basic 9 — Compare a tempting alternative

**Goal.** Compute an absolute and relative gap, because a lower score matters only if the improvement is meaningful. We build it in 2 steps.

In [ ]:
baseline_b9 = 0.331  # Lesson baseline decision score.
flexible_b9 = 0.371  # Lesson tempting flexible alternative score.
gap_b9 = flexible_b9 - baseline_b9  # Absolute evidence gap.
relative_b9 = gap_b9 / flexible_b9  # Gap as a fraction of the alternative.
print("gap:", round(gap_b9, 3), "relative gap:", round(relative_b9, 3))  # Inspect comparison.
assert round(gap_b9, 3) == 0.040  # Verify lesson gap.

▶ What you'll see: the baseline wins by 0.040, about 10.8% of the alternative score.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a score comparison.
plt.bar(["baseline", "flexible"], [baseline_b9, flexible_b9], color=["seagreen", "crimson"])  # Compare decision scores.
plt.title("Basic 9: compare full scores")  # Title plot.
plt.ylabel("lower is better")  # Label y-axis.
plt.show()  # Display bars.

▶ What you'll see: the flexible model loses after its full decision score is counted.

👀 Takeaway: a validation or decision gap is evidence; if it is tiny, the apparent winner may be noise.

### Basic 10 — Stabilize the decision score

**Goal.** Apply the lesson's 20% stabilization improvement, because regularization can lower future-facing decision cost even when it constrains fit. We build it in 2 steps.

In [ ]:
score_b10 = 0.331  # Start from the lesson baseline score.
stable_b10 = 0.80 * score_b10  # Apply a 20% stabilizing reduction.
options_b10 = np.array([score_b10, 0.371, stable_b10])  # Compare baseline, flexible, and stabilized scores.
print("scores:", np.round(options_b10, 3))  # Inspect the three choices.
assert round(stable_b10, 3) == 0.265  # Verify lesson stable score.

▶ What you'll see: the stabilized option has the smallest score in the toy decision.

In [ ]:
winner_b10 = int(np.argmin(options_b10))  # Find the lowest score.
print("winning index:", winner_b10, "winning score:", round(float(options_b10[winner_b10]), 3))  # Inspect final decision.
plt.figure(figsize=(4, 3))  # Plot final decision.
plt.bar(["baseline", "flexible", "stabilized"], options_b10, color=["gray", "crimson", "seagreen"])  # Compare all options.
plt.title("Basic 10: final score comparison")  # Title plot.
plt.xticks(rotation=12)  # Rotate labels for readability.
plt.ylabel("decision score")  # Label y-axis.
plt.show()  # Display bars.

▶ What you'll see: the stabilized bar is lowest, so it is the toy winner.

👀 Takeaway: regularization is a decision mechanism, not decoration after training.

## 🟡 Easy

### Easy 1 — Fit and plot KRR on the toy curve

**Goal.** Train a complete KRR model and draw its prediction curve, because the full algorithm is kernel matrix, ridge solve, then similarity prediction. We build it in 3 steps.

In [ ]:
X_e1 = np.linspace(-3, 3, 9)[:, None]  # Create training inputs.
y_e1 = np.sin(X_e1[:, 0]) + 0.18 * X_e1[:, 0]  # Create nonlinear targets.
gamma_e1 = 0.7  # Choose RBF locality.
lam_e1 = 0.12  # Choose ridge strength.
alpha_e1, K_e1 = krr_fit(X_e1, y_e1, gamma_e1, lam_e1)  # Fit KRR.
print("alpha norm:", round(float(np.linalg.norm(alpha_e1)), 3))  # Inspect weight size.

▶ What you'll see: a finite alpha norm, meaning the regularized solve produced stable weights.

In [ ]:
grid_e1 = np.linspace(-3.4, 3.4, 160)[:, None]  # Create a plotting grid.
pred_e1 = krr_predict(grid_e1, X_e1, alpha_e1, gamma_e1)  # Predict along the grid.
train_pred_e1 = K_e1 @ alpha_e1  # Predict at training points.
rmse_e1 = float(np.sqrt(np.mean((train_pred_e1 - y_e1) ** 2)))  # Compute training RMSE.
print("train RMSE:", round(rmse_e1, 4))  # Inspect fit quality.
assert rmse_e1 < 0.15  # Verify the toy model fits the smooth signal.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a fit plot.
plt.plot(grid_e1[:, 0], pred_e1, color="seagreen", label="KRR")  # Draw KRR curve.
plt.scatter(X_e1[:, 0], y_e1, color="black", s=24, label="data")  # Draw training points.
plt.title("Easy 1: KRR fit")  # Title plot.
plt.legend(); plt.show()  # Display fit.

▶ What you'll see: the prediction curve follows the nonlinear data while staying smooth.

👀 Takeaway: complete KRR is a small pipeline: kernel similarities, regularized solve, kernel prediction.

### Easy 2 — Sweep lambda for smoothness

**Goal.** Compare ridge strengths at fixed bandwidth, because λ controls how much the model trusts flexibility. We build it in 3 steps.

In [ ]:
X_e2 = np.linspace(-3, 3, 9)[:, None]  # Define training inputs.
y_e2 = np.sin(X_e2[:, 0]) + 0.18 * X_e2[:, 0]  # Define targets.
lams_e2 = np.array([0.01, 0.12, 1.0])  # Try weak, medium, and strong ridge.
grid_e2 = np.linspace(-3.3, 3.3, 140)[:, None]  # Create prediction grid.
print("lambda grid:", lams_e2)  # Inspect candidates.

▶ What you'll see: three ridge strengths are ready for comparison.

In [ ]:
curves_e2 = []  # Store predictions for each lambda.
norms_e2 = []  # Store alpha norms for each lambda.
for lam_e2 in lams_e2:  # Fit one model per lambda.
    alpha_e2, K_e2 = krr_fit(X_e2, y_e2, gamma=0.7, lam=lam_e2)  # Fit KRR.
    curves_e2.append(krr_predict(grid_e2, X_e2, alpha_e2, 0.7))  # Predict grid.
    norms_e2.append(float(np.linalg.norm(alpha_e2)))  # Save weight size.
print("alpha norms:", np.round(norms_e2, 3))  # Inspect shrinkage.
assert norms_e2[0] > norms_e2[-1]  # Stronger ridge shrinks alpha here.

In [ ]:
plt.figure(figsize=(5, 3))  # Create comparison plot.
for lam_e2, curve_e2 in zip(lams_e2, curves_e2):  # Plot each curve.
    plt.plot(grid_e2[:, 0], curve_e2, label=f"λ={lam_e2}")  # Label by lambda.
plt.scatter(X_e2[:, 0], y_e2, color="black", s=18)  # Show training data.
plt.title("Easy 2: ridge strength")  # Title plot.
plt.legend(); plt.show()  # Display curves.

▶ What you'll see: larger λ produces a more damped, smoother curve with smaller alpha weights.

👀 Takeaway: λ is the explicit regularization knob in the KRR linear system.

### Easy 3 — Sweep gamma for locality

**Goal.** Compare RBF bandwidths at fixed λ, because gamma decides how far each training example's influence reaches. We build it in 3 steps.

In [ ]:
X_e3 = np.linspace(-3, 3, 9)[:, None]  # Define training inputs.
y_e3 = np.sin(X_e3[:, 0]) + 0.18 * X_e3[:, 0]  # Define targets.
gammas_e3 = np.array([0.15, 0.7, 4.0])  # Broad, medium, and local kernels.
grid_e3 = np.linspace(-3.3, 3.3, 140)[:, None]  # Prediction grid.
print("gamma grid:", gammas_e3)  # Inspect candidates.

▶ What you'll see: three locality settings are ready.

In [ ]:
curves_e3 = []  # Store grid predictions.
band_sums_e3 = []  # Store average row sums as an influence-width proxy.
for gamma_e3 in gammas_e3:  # Fit each bandwidth.
    alpha_e3, K_e3 = krr_fit(X_e3, y_e3, gamma_e3, lam=0.12)  # Fit KRR.
    curves_e3.append(krr_predict(grid_e3, X_e3, alpha_e3, gamma_e3))  # Predict grid.
    band_sums_e3.append(float(np.mean(K_e3.sum(axis=1))))  # Larger means broader influence.
print("average kernel row sums:", np.round(band_sums_e3, 3))  # Inspect locality proxy.
assert band_sums_e3[0] > band_sums_e3[-1]  # Small gamma has broader similarity rows.

In [ ]:
plt.figure(figsize=(5, 3))  # Create bandwidth plot.
for gamma_e3, curve_e3 in zip(gammas_e3, curves_e3):  # Plot each curve.
    plt.plot(grid_e3[:, 0], curve_e3, label=f"γ={gamma_e3}")  # Label by gamma.
plt.scatter(X_e3[:, 0], y_e3, color="black", s=18)  # Show data.
plt.title("Easy 3: bandwidth/locality")  # Title plot.
plt.legend(); plt.show()  # Display curves.

▶ What you'll see: small gamma spreads influence broadly; large gamma makes more local bumps.

👀 Takeaway: gamma changes the geometry before ridge regression ever solves for alpha.

### Easy 4 — Validate candidate hyperparameters

**Goal.** Choose a KRR setting with held-out error plus a small complexity cost, because the lesson's decision score is raw fit plus cost. We build it in 4 steps.

In [ ]:
X_e4 = np.linspace(-3, 3, 9)[:, None]  # Training inputs.
y_e4 = np.sin(X_e4[:, 0]) + 0.18 * X_e4[:, 0]  # Training targets.
X_val_e4 = np.array([[-2.6], [-1.1], [0.4], [1.9], [2.8]])  # Validation inputs.
y_val_e4 = np.sin(X_val_e4[:, 0]) + 0.18 * X_val_e4[:, 0]  # Validation targets.
candidates_e4 = [(0.15, 0.12), (0.7, 0.12), (4.0, 0.12)]  # Candidate (gamma, lambda) pairs.
print("candidates:", candidates_e4)  # Inspect search grid.

▶ What you'll see: validation will compare broad, medium, and local kernels.

In [ ]:
val_mse_e4 = []  # Store raw validation losses.
costs_e4 = []  # Store complexity costs.
for gamma_e4, lam_e4 in candidates_e4:  # Evaluate candidates.
    alpha_e4, K_e4 = krr_fit(X_e4, y_e4, gamma_e4, lam_e4)  # Fit model.
    pred_val_e4 = krr_predict(X_val_e4, X_e4, alpha_e4, gamma_e4)  # Predict validation set.
    val_mse_e4.append(float(np.mean((pred_val_e4 - y_val_e4) ** 2)))  # Raw validation MSE.
    costs_e4.append(0.01 * float(alpha_e4 @ K_e4 @ alpha_e4))  # Complexity proxy.
print("validation MSE:", np.round(val_mse_e4, 4))  # Inspect raw fit.
print("costs:", np.round(costs_e4, 4))  # Inspect cost term.

In [ ]:
scores_e4 = np.array(val_mse_e4) + np.array(costs_e4)  # Add cost to validation fit.
best_e4 = int(np.argmin(scores_e4))  # Select the lowest final score.
print("scores:", np.round(scores_e4, 4), "best:", candidates_e4[best_e4])  # Inspect winner.
assert scores_e4[best_e4] == np.min(scores_e4)  # Verify selection.

In [ ]:
plt.figure(figsize=(5, 3))  # Plot final decision scores.
plt.bar([f"γ={g}" for g, _ in candidates_e4], scores_e4, color=["seagreen" if i == best_e4 else "gray" for i in range(len(scores_e4))])  # Highlight winner.
plt.title("Easy 4: validation score + cost")  # Title plot.
plt.ylabel("decision score")  # Label y-axis.
plt.show()  # Display bars.

▶ What you'll see: the chosen hyperparameter pair minimizes the full score, not just one isolated number.

👀 Takeaway: validation plus cost operationalizes the regularization warning from the lesson text.

### Easy 5 — Compare KRR with a linear ridge baseline

**Goal.** Fit an explicit linear ridge model and compare validation error, because kernels are useful only when their added flexibility helps future data. We build it in 4 steps.

In [ ]:
X_e5 = np.linspace(-3, 3, 9)[:, None]  # Training inputs.
y_e5 = np.sin(X_e5[:, 0]) + 0.18 * X_e5[:, 0]  # Training targets.
X_val_e5 = np.array([[-2.6], [-1.1], [0.4], [1.9], [2.8]])  # Validation inputs.
y_val_e5 = np.sin(X_val_e5[:, 0]) + 0.18 * X_val_e5[:, 0]  # Validation targets.
Phi_e5 = np.c_[np.ones(len(X_e5)), X_e5[:, 0]]  # Linear features with intercept.
Phi_val_e5 = np.c_[np.ones(len(X_val_e5)), X_val_e5[:, 0]]  # Validation linear features.
print("linear feature shape:", Phi_e5.shape)  # Inspect explicit baseline design.

▶ What you'll see: the baseline has only intercept and slope features.

In [ ]:
lam_e5 = 0.12  # Ridge penalty for both models.
pen_e5 = np.diag([0.0, lam_e5])  # Do not penalize intercept.
w_e5 = np.linalg.solve(Phi_e5.T @ Phi_e5 + pen_e5, Phi_e5.T @ y_e5)  # Fit linear ridge.
pred_lin_e5 = Phi_val_e5 @ w_e5  # Predict validation data.
rmse_lin_e5 = float(np.sqrt(np.mean((pred_lin_e5 - y_val_e5) ** 2)))  # Validation RMSE.
print("linear ridge weights:", np.round(w_e5, 3), "RMSE:", round(rmse_lin_e5, 3))  # Inspect baseline.

In [ ]:
alpha_e5, K_e5 = krr_fit(X_e5, y_e5, gamma=0.7, lam=lam_e5)  # Fit kernel ridge.
pred_krr_e5 = krr_predict(X_val_e5, X_e5, alpha_e5, 0.7)  # Predict validation data.
rmse_krr_e5 = float(np.sqrt(np.mean((pred_krr_e5 - y_val_e5) ** 2)))  # Validation RMSE.
print("KRR RMSE:", round(rmse_krr_e5, 3))  # Inspect KRR validation error.
assert rmse_krr_e5 < rmse_lin_e5  # The nonlinear kernel wins on this curved signal.

In [ ]:
plt.figure(figsize=(4, 3))  # Create model comparison plot.
plt.bar(["linear ridge", "kernel ridge"], [rmse_lin_e5, rmse_krr_e5], color=["gray", "seagreen"])  # Compare validation errors.
plt.title("Easy 5: validation RMSE")  # Title plot.
plt.ylabel("RMSE lower is better")  # Label y-axis.
plt.xticks(rotation=10)  # Rotate labels.
plt.show()  # Display bars.

▶ What you'll see: KRR has lower validation error because the target is curved.

👀 Takeaway: the kernel should earn its complexity by improving held-out performance over simpler ridge.

## 🔴 Advanced

### Advanced 1 — Build polynomial KRR from scratch

**Goal.** Swap the RBF kernel for a polynomial kernel, because KRR is a framework for any valid similarity function. We build it in 4 steps.

In [ ]:
X_a1 = np.linspace(-2, 2, 11)[:, None]  # Create one-dimensional inputs.
y_a1 = 0.4 * X_a1[:, 0] ** 2 - 0.2 * X_a1[:, 0] + 0.1  # Quadratic target.
def poly_kernel_a1(A_a1, B_a1, degree_a1=2, c_a1=1.0):  # Define polynomial kernel locally.
    return (A_a1 @ B_a1.T + c_a1) ** degree_a1  # Compute (x z + c)^degree.
print("target range:", round(float(y_a1.min()), 3), "to", round(float(y_a1.max()), 3))  # Inspect signal scale.

▶ What you'll see: the target is a simple curved quadratic.

In [ ]:
K_a1 = poly_kernel_a1(X_a1, X_a1, degree_a1=2, c_a1=1.0)  # Build polynomial Gram matrix.
lam_a1 = 0.05  # Choose ridge strength.
alpha_a1 = np.linalg.solve(K_a1 + lam_a1 * np.eye(len(X_a1)), y_a1)  # Fit polynomial KRR.
fit_a1 = K_a1 @ alpha_a1  # Training predictions.
rmse_a1 = float(np.sqrt(np.mean((fit_a1 - y_a1) ** 2)))  # Training RMSE.
print("train RMSE:", round(rmse_a1, 5))  # Inspect fit.
assert rmse_a1 < 0.03  # Degree-2 kernel fits a quadratic well.

In [ ]:
grid_a1 = np.linspace(-2.3, 2.3, 140)[:, None]  # Plotting grid.
pred_a1 = poly_kernel_a1(grid_a1, X_a1, degree_a1=2, c_a1=1.0) @ alpha_a1  # Predict grid.
print("alpha norm:", round(float(np.linalg.norm(alpha_a1)), 3))  # Inspect coefficient size.

In [ ]:
plt.figure(figsize=(5, 3))  # Create polynomial fit plot.
plt.plot(grid_a1[:, 0], pred_a1, color="purple", label="poly KRR")  # Draw prediction curve.
plt.scatter(X_a1[:, 0], y_a1, color="black", s=20, label="data")  # Draw training points.
plt.title("Advanced 1: polynomial kernel ridge")  # Title plot.
plt.legend(); plt.show()  # Display figure.

▶ What you'll see: a smooth parabola-like curve matching the quadratic data.

👀 Takeaway: the KRR solve is unchanged; only the kernel function changes the feature space.

### Advanced 2 — Inspect the RKHS norm cost proxy

**Goal.** Compute `alphaᵀKalpha`, because the KRR complexity term can be read from the dual weights and kernel matrix. We build it in 4 steps.

In [ ]:
X_a2 = np.linspace(-3, 3, 9)[:, None]  # Training inputs.
y_a2 = np.sin(X_a2[:, 0]) + 0.18 * X_a2[:, 0]  # Training targets.
lams_a2 = np.array([0.02, 0.12, 0.6])  # Ridge strengths to inspect.
complexities_a2 = []  # Store alpha^T K alpha.
train_rmse_a2 = []  # Store training RMSE.
print("lambdas:", lams_a2)  # Inspect settings.

▶ What you'll see: weak to strong regularization values are ready.

In [ ]:
for lam_a2 in lams_a2:  # Fit each ridge strength.
    alpha_a2, K_a2 = krr_fit(X_a2, y_a2, gamma=0.7, lam=lam_a2)  # Fit KRR.
    complexities_a2.append(float(alpha_a2 @ K_a2 @ alpha_a2))  # RKHS-norm-style cost.
    train_rmse_a2.append(float(np.sqrt(np.mean((K_a2 @ alpha_a2 - y_a2) ** 2))))  # Training RMSE.
print("complexities:", np.round(complexities_a2, 3))  # Inspect complexity proxy.
print("train RMSE:", np.round(train_rmse_a2, 4))  # Inspect fit.

In [ ]:
assert complexities_a2[0] >= complexities_a2[-1]  # Stronger ridge should shrink the function norm here.
assert train_rmse_a2[0] <= train_rmse_a2[-1]  # Stronger ridge usually increases training error.
print("tradeoff verified")  # Confirm expected direction.

In [ ]:
plt.figure(figsize=(5, 3))  # Create two-axis-style comparison with normalized values.
plt.plot(lams_a2, np.array(complexities_a2) / max(complexities_a2), marker="o", label="scaled complexity")  # Complexity curve.
plt.plot(lams_a2, np.array(train_rmse_a2) / max(train_rmse_a2), marker="s", label="scaled train RMSE")  # Fit curve.
plt.title("Advanced 2: fit versus function size")  # Title plot.
plt.xlabel("λ")  # Label x-axis.
plt.legend(); plt.show()  # Display curves.

▶ What you'll see: λ trades off smaller function norm against larger training error.

👀 Takeaway: regularization cost has a concrete dual expression, not just a vague preference for smoothness.

### Advanced 3 — Leave-one-out style sensitivity

**Goal.** Hide each training point once and measure prediction error, because KRR should be judged by future-like examples rather than only training fit. We build it in 4 steps.

In [ ]:
X_a3 = np.linspace(-3, 3, 9)[:, None]  # Full input set.
y_a3 = np.sin(X_a3[:, 0]) + 0.18 * X_a3[:, 0]  # Full target set.
gamma_a3 = 0.7  # Fixed bandwidth.
lam_a3 = 0.12  # Fixed ridge.
errors_a3 = []  # Store one held-out error per point.
print("number of held-out trials:", len(X_a3))  # Inspect validation count.

▶ What you'll see: each of the nine points will become validation once.

In [ ]:
for i_a3 in range(len(X_a3)):  # Leave out each index.
    keep_a3 = np.arange(len(X_a3)) != i_a3  # Training mask for this fold.
    alpha_a3, K_train_a3 = krr_fit(X_a3[keep_a3], y_a3[keep_a3], gamma_a3, lam_a3)  # Fit without the held-out point.
    pred_i_a3 = float(krr_predict(X_a3[i_a3:i_a3+1], X_a3[keep_a3], alpha_a3, gamma_a3))  # Predict held-out point.
    errors_a3.append(pred_i_a3 - y_a3[i_a3])  # Store signed error.
print("held-out errors:", np.round(errors_a3, 3))  # Inspect fold errors.

In [ ]:
rmse_a3 = float(np.sqrt(np.mean(np.array(errors_a3) ** 2)))  # Aggregate held-out RMSE.
print("leave-one-out style RMSE:", round(rmse_a3, 3))  # Inspect validation summary.
assert rmse_a3 < 0.35  # Verify the smooth toy signal is predicted reasonably.

In [ ]:
plt.figure(figsize=(5, 3))  # Create held-out error plot.
plt.bar(range(len(errors_a3)), errors_a3, color="steelblue")  # One signed error per held-out point.
plt.axhline(0, color="black", linewidth=0.8)  # Zero-error reference.
plt.title("Advanced 3: held-out error by point")  # Title plot.
plt.xlabel("held-out index"); plt.ylabel("prediction error")  # Label axes.
plt.show()  # Display bars.

▶ What you'll see: edge points often have larger errors because they have neighbors on only one side.

👀 Takeaway: validation exposes where the similarity smoother extrapolates weakly.

### Advanced 4 — Compare condition numbers across gamma and lambda

**Goal.** Map numerical stability over hyperparameters, because narrow kernels and tiny λ can make the solve sensitive. We build it in 4 steps.

In [ ]:
X_a4 = np.linspace(-3, 3, 13)[:, None]  # Slightly larger grid of training inputs.
gammas_a4 = np.array([0.05, 0.3, 1.5, 6.0])  # Bandwidth settings.
lams_a4 = np.array([0.001, 0.03, 0.3])  # Ridge settings.
conds_a4 = np.zeros((len(lams_a4), len(gammas_a4)))  # Storage for condition numbers.
print("grid shape:", conds_a4.shape)  # Inspect heatmap dimensions.

▶ What you'll see: the stability grid has lambda rows and gamma columns.

In [ ]:
for i_a4, lam_a4 in enumerate(lams_a4):  # Loop over ridge values.
    for j_a4, gamma_a4 in enumerate(gammas_a4):  # Loop over bandwidth values.
        K_a4 = rbf_kernel(X_a4, X_a4, gamma_a4)  # Build kernel matrix.
        conds_a4[i_a4, j_a4] = np.linalg.cond(K_a4 + lam_a4 * np.eye(len(X_a4)))  # Condition regularized system.
print("condition grid:\n", np.round(conds_a4, 1))  # Inspect numbers.
assert np.all(conds_a4[0] >= conds_a4[-1])  # Larger lambda improves conditioning for each gamma here.

In [ ]:
log_conds_a4 = np.log10(conds_a4)  # Use log scale so the heatmap is readable.
print("log10 condition grid:\n", np.round(log_conds_a4, 2))  # Inspect log-scaled stability.

In [ ]:
plt.figure(figsize=(5, 3))  # Create stability heatmap.
plt.imshow(log_conds_a4, cmap="magma", aspect="auto")  # Plot log condition numbers.
plt.colorbar(label="log10 condition")  # Add color scale.
plt.xticks(range(len(gammas_a4)), gammas_a4)  # Label gamma columns.
plt.yticks(range(len(lams_a4)), lams_a4)  # Label lambda rows.
plt.xlabel("gamma"); plt.ylabel("lambda")  # Label axes.
plt.title("Advanced 4: solve stability")  # Title plot.
plt.show()  # Display heatmap.

▶ What you'll see: increasing λ generally darkens the heatmap by lowering the condition number.

👀 Takeaway: λ is both a statistical regularizer and a numerical stabilizer for the kernel system.

### Advanced 5 — End-to-end model selection with a baseline

**Goal.** Run a small train/validation selection loop against a linear baseline, because the final KRR decision should include fit, cost, and a simpler alternative. We build it in 5 steps.

In [ ]:
X_a5 = np.linspace(-3, 3, 15)[:, None]  # Full input set.
y_a5 = np.sin(X_a5[:, 0]) + 0.18 * X_a5[:, 0]  # Full smooth targets.
train_a5 = np.arange(len(X_a5)) % 3 != 0  # Deterministic train split.
val_a5 = ~train_a5  # Validation split.
print("train/val sizes:", int(train_a5.sum()), int(val_a5.sum()))  # Inspect split sizes.

▶ What you'll see: a small but deterministic validation protocol.

In [ ]:
Phi_train_a5 = np.c_[np.ones(train_a5.sum()), X_a5[train_a5, 0]]  # Linear training features.
Phi_val_a5 = np.c_[np.ones(val_a5.sum()), X_a5[val_a5, 0]]  # Linear validation features.
w_a5 = np.linalg.solve(Phi_train_a5.T @ Phi_train_a5 + np.diag([0.0, 0.1]), Phi_train_a5.T @ y_a5[train_a5])  # Fit linear ridge.
base_pred_a5 = Phi_val_a5 @ w_a5  # Predict validation set.
base_rmse_a5 = float(np.sqrt(np.mean((base_pred_a5 - y_a5[val_a5]) ** 2)))  # Baseline RMSE.
print("linear baseline RMSE:", round(base_rmse_a5, 3))  # Inspect baseline.

In [ ]:
candidates_a5 = [(0.15, 0.05), (0.7, 0.12), (2.5, 0.12), (4.0, 0.3)]  # Candidate KRR settings.
records_a5 = []  # Store (score, rmse, cost, gamma, lambda).
for gamma_a5, lam_a5 in candidates_a5:  # Evaluate each KRR candidate.
    alpha_a5, K_a5 = krr_fit(X_a5[train_a5], y_a5[train_a5], gamma_a5, lam_a5)  # Fit train split.
    pred_a5 = krr_predict(X_a5[val_a5], X_a5[train_a5], alpha_a5, gamma_a5)  # Predict validation split.
    rmse_a5 = float(np.sqrt(np.mean((pred_a5 - y_a5[val_a5]) ** 2)))  # Validation RMSE.
    cost_a5 = 0.01 * float(alpha_a5 @ K_a5 @ alpha_a5)  # Complexity cost.
    records_a5.append((rmse_a5 + cost_a5, rmse_a5, cost_a5, gamma_a5, lam_a5))  # Store full score.
print("records score/rmse/cost/gamma/lambda:\n", np.round(np.array(records_a5), 4))  # Inspect candidates.

In [ ]:
records_a5 = np.array(records_a5, dtype=float)  # Convert to array for selection.
best_idx_a5 = int(np.argmin(records_a5[:, 0]))  # Choose lowest score.
best_score_a5 = float(records_a5[best_idx_a5, 0])  # Read selected score.
print("best KRR setting:", tuple(records_a5[best_idx_a5, 3:5]), "score:", round(best_score_a5, 3))  # Report winner.
assert best_score_a5 < base_rmse_a5  # Verify selected KRR beats the baseline's raw validation RMSE on this toy split.

In [ ]:
plt.figure(figsize=(5, 3))  # Create final comparison plot.
labels_a5 = [f"γ={g},λ={l}" for _, _, _, g, l in records_a5]  # Candidate labels.
plt.bar(range(len(records_a5)), records_a5[:, 0], color=["seagreen" if i == best_idx_a5 else "gray" for i in range(len(records_a5))])  # Plot KRR scores.
plt.axhline(base_rmse_a5, color="crimson", linestyle="--", label="linear baseline RMSE")  # Add baseline reference.
plt.xticks(range(len(records_a5)), labels_a5, rotation=25, ha="right")  # Label candidates.
plt.ylabel("decision score / RMSE")  # Label y-axis.
plt.title("Advanced 5: select KRR against baseline")  # Title plot.
plt.legend(); plt.tight_layout(); plt.show()  # Display final chart.

▶ What you'll see: the best KRR bar falls below the linear baseline reference on this curved task.

👀 Takeaway: a kernel model is worth carrying forward only when its validated score justifies the added flexibility.